# Evaluación Sumativa Unidad 2 — Minería de Datos (`IF308IINF`)
**Instituto Profesional San Sebastián (IPSS) | Carrera: Ingeniería en Informática**  
**Asignatura:** Minería de Datos (`IF308IINF`) | **Sección:** `T3-26-(51)`  
**Estudiante:** Gabriel Ignacio Báez Báez  
**Docente:** Nelson Carvallo Suma de Villa  

---

### Marco Metodológico y Resumen de Modelos Aplicados
En esta evaluación se abordan tres paradigmas fundamentales del aprendizaje automático en Python sobre conjuntos de datos reales disponibles en Kaggle: **clasificación supervisada multiclase** (*Iris Dataset*), **agrupamiento no supervisado** (*Mall Customers Dataset*) y **regresión predictiva** (*Boston Housing Dataset*).

Con el objetivo de alcanzar el estándar de **Nivel Sobresaliente**, cada desafío ejecuta tanto el modelo base exigido por la pauta como un flujo completo de preprocesamiento reproducible (`SimpleImputer`, `StandardScaler`, `Pipeline`), reducción de dimensionalidad (`PCA`), optimización de hiperparámetros con validación cruzada (`GridSearchCV`, `StratifiedKFold`, `KFold`), comparación frente a algoritmos alternativos y evaluación crítica con métricas complementarias:

| Desafío | Dataset | Paradigma | Modelo Base (Pauta) | Optimización y Modelos Comparativos | Métricas Aplicadas |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1. Clasificación** | **Iris Dataset** (`150 × 4`) | Supervisado (Multiclase) | `KNN (k = 3)` | `Pipeline + StandardScaler`, `GridSearchCV` (`k` óptimo), `PCA 2D`, `Random Forest` y `SVM (RBF)` | `Accuracy`, `Precision`, `Recall`, `F1-Score`, Matriz de Confusión, `CV-5` |
| **2. Clustering** | **Mall Customers** (`200 × 5`) | No Supervisado | `K-Means (k = 2, 3)` | `StandardScaler`, Método del Codo, Índice de Silueta (`k = 5`), Clustering Jerárquico Ward, `DBSCAN`, extensión 3D (`PCA 2D`) | Inercia (`WCSS`), `Silhouette Score`, `Davies-Bouldin`, Visualización 2D |
| **3. Regresión** | **Boston Housing** (`506 × 14`) | Supervisado (Continuo) | Regresión Lineal Simple (`RM`) | `SimpleImputer`, Regresión Múltiple, `RidgeCV`, `LassoCV` y `Random Forest Regressor` (`GridSearchCV`) | $R^2$, $R^2$ ajustado, `MAE`, `MSE`, `RMSE` y Análisis de Residuos |

### Nota Metodológica de Reproducibilidad y Homologación
1. **Carga reproducible (local y Google Colab):** Los archivos `Mall_Customers.csv` y `BostonHousing.csv` se leen de manera prioritaria desde la carpeta local del repositorio y cuentan con respaldo automático mediante URL pública para ejecución directa en Google Colab.
2. **Homologación de columnas:** En *Mall Customers*, distintas copias de Kaggle nombran la columna de género como `Genre` o `Gender`; el cargador la estandariza explícitamente como `Gender`. En *Boston Housing*, todas las columnas se normalizan a mayúsculas (`CRIM`, `RM`, `LSTAT`, `MEDV`, etc.) para consistencia analítica.
3. **Nota ética sobre `load_boston()`:** La función `sklearn.datasets.load_boston()` fue removida a partir de `scikit-learn 1.2` debido a consideraciones éticas en la construcción histórica de la variable `B` (Harrison & Rubinfeld, 1978; Pedregosa et al., 2011). Por ello, el dataset exigido por la pauta se ingesta desde CSV y en la sección de conclusiones se discute críticamente esta limitación y las alternativas recomendadas (*California Housing* / *Ames Housing*).

## 0. Configuración del Entorno, Librerías y Funciones de Carga
Se importan las bibliotecas científicas estándar (`numpy`, `pandas`, `matplotlib`, `seaborn`, `scipy`, `scikit-learn`), se fija la semilla aleatoria (`RANDOM_STATE = 42`) para asegurar reproducibilidad determinista y se definen funciones modulares de carga con respaldo en línea.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.datasets import load_iris
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    KFold,
    cross_val_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
    silhouette_score,
    davies_bouldin_score,
    r2_score,
    mean_absolute_error,
    mean_squared_error,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120


def cargar_mall_customers(local_path: str = "Mall_Customers.csv") -> pd.DataFrame:
    '''Carga Mall Customers desde disco local o desde mirrors públicos de GitHub.'''
    urls = [
        "https://raw.githubusercontent.com/krishnaik06/DBSCAN-Algorithm/master/Mall_Customers.csv",
        "https://raw.githubusercontent.com/profsarang/ThinkingDataScience/main/data/Mall_Customers.csv",
    ]
    if os.path.exists(local_path):
        df = pd.read_csv(local_path)
        origen = f"archivo local '{local_path}'"
    else:
        df = None
        for url in urls:
            try:
                df = pd.read_csv(url)
                origen = f"URL pública ({url})"
                break
            except Exception:
                continue
        if df is None:
            raise FileNotFoundError("No fue posible cargar Mall_Customers.csv ni localmente ni vía URL.")

    df.columns = [c.strip() for c in df.columns]
    if "Genre" in df.columns and "Gender" not in df.columns:
        df = df.rename(columns={"Genre": "Gender"})
    print(f"[Mall Customers] Cargado correctamente desde {origen} -> {df.shape[0]} filas x {df.shape[1]} columnas.")
    return df


def cargar_boston_housing(local_path: str = "BostonHousing.csv") -> pd.DataFrame:
    '''Carga Boston Housing desde disco local, mirror GitHub o fuente original StatLib CMU.'''
    url_csv = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
    url_statlib = "https://lib.stat.cmu.edu/datasets/boston"

    if os.path.exists(local_path):
        df = pd.read_csv(local_path)
        origen = f"archivo local '{local_path}'"
    else:
        try:
            df = pd.read_csv(url_csv)
            origen = f"URL pública ({url_csv})"
        except Exception:
            raw_df = pd.read_csv(url_statlib, sep=r"\s+", skiprows=22, header=None)
            data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
            target = raw_df.values[1::2, 2]
            cols = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT"]
            df = pd.DataFrame(data, columns=cols)
            df["MEDV"] = target
            origen = f"CMU StatLib ({url_statlib})"

    df.columns = [c.strip().upper() for c in df.columns]
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    print(f"[Boston Housing] Cargado correctamente desde {origen} -> {df.shape[0]} filas x {df.shape[1]} columnas.")
    return df


print("Entorno configurado correctamente. Semilla aleatoria fijada en:", RANDOM_STATE)

---
## Desafío 1: Clasificación Supervisada — *Iris Dataset*

### 1.a. Exploración de los Datos (`EDA`) y Variables del Dataset
El conjunto de datos **Iris** (Fisher, 1936) es el estándar clásico de clasificación multiclase balanceada. Contiene **$150$ observaciones** distribuidas equitativamente en **$3$ especies botánicas** ($50$ muestras cada una: *Setosa*, *Versicolor* y *Virginica*) y **$4$ variables predictoras numéricas continuas** medidas en centímetros:

| Variable | Descripción | Unidad | Rol en el Modelo | Relevancia Analítica Esperada |
| :--- | :--- | :---: | :--- | :--- |
| `sepal length (cm)` | Longitud del sépalo (hoja externa protectora de la flor) | cm | Predictor numérico | Presenta solapamiento moderado entre *Versicolor* y *Virginica*; correlación positiva con el tamaño del pétalo. |
| `sepal width (cm)` | Ancho del sépalo | cm | Predictor numérico | Posee la mayor dispersión intraclase y menor poder discriminativo individual; *Setosa* suele registrar valores más altos. |
| `petal length (cm)` | Longitud del pétalo | cm | Predictor numérico | **Variable discriminativa principal**; separa linealmente a *Iris setosa* del resto de especies. |
| `petal width (cm)` | Ancho del pétalo | cm | Predictor numérico | Altamente correlacionada con la longitud del pétalo; clave para trazar la frontera entre *Versicolor* y *Virginica*. |
| `target` / `species` | Especie botánica (`0: setosa`, `1: versicolor`, `2: virginica`) | Categórica | Variable objetivo | Distribución perfectamente balanceada ($33.3\%$ por clase). |

### 1.b. Preprocesamiento, División Train/Test (`train_test_split`) y Reducción Dimensional (`PCA`)
* **¿Por qué dividir en entrenamiento y prueba?** Si entrenamos y evaluamos un algoritmo sobre los mismos registros, el modelo podría memorizar el ruido de los datos (*overfitting*) y mostrar un rendimiento artificialmente optimista. Separar los datos mediante `train_test_split(..., test_size=0.20, random_state=42, stratify=y)` reserva un **$80\%$ ($120$ muestras)** para el aprendizaje de patrones y un **$20\%$ ($30$ muestras independientes)** para medir objetivamente la **capacidad de generalización** ante flores no vistas (James et al., 2021).
* **Estratificación (`stratify=y`):** Garantiza que tanto el conjunto de entrenamiento como el de prueba mantengan exactamente la proporción $1/3$ de cada especie ($40$ por clase en *train* y $10$ por clase en *test*), evitando sesgos muestrales en conjuntos pequeños.
* **Escalamiento dentro de `Pipeline` (evitando *Data Leakage*):** Como *k-Nearest Neighbors* (`KNN`) clasifica midiendo distancias euclidianas, atributos con rangos mayores dominarían el cálculo. Encapsular `StandardScaler()` dentro de un `Pipeline` ajusta la media y desviación estándar **solo con `X_train`**, evitando fuga de información del conjunto de prueba. Complementariamente, se aplica **Análisis de Componentes Principales (`PCA`)** a $2$ dimensiones para verificar la separabilidad global en un plano ortogonal.

In [ ]:
# 1. Carga y exploración del Iris Dataset
iris = load_iris(as_frame=True)
df_iris = iris.frame.copy()
df_iris["species"] = df_iris["target"].map(dict(enumerate(iris.target_names)))

print(f"Dimensiones: {df_iris.shape[0]} filas x {df_iris.shape[1]} columnas")
print("Valores nulos por columna:\n", df_iris.isnull().sum().to_dict())
print("Filas duplicadas:", df_iris.duplicated().sum())
print("\nDistribución de clases (species):")
print(df_iris["species"].value_counts())

print("\nEstadísticas descriptivas globales:")
print(df_iris[iris.feature_names].describe().round(3))

print("\nPromedios morfológicos por especie (cm):")
print(df_iris.groupby("species")[iris.feature_names].mean().round(3))

corr_iris = df_iris[iris.feature_names].corr()
print("\nMatriz de correlación de Pearson:")
print(corr_iris.round(3))

# Figura 1: Heatmap de correlaciones y dispersión de pétalos por especie
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

sns.heatmap(corr_iris, annot=True, cmap="Blues", fmt=".2f", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Matriz de Correlación — Iris Dataset")

sns.scatterplot(
    data=df_iris,
    x="petal length (cm)",
    y="petal width (cm)",
    hue="species",
    style="species",
    s=75,
    palette="viridis",
    ax=axes[1],
)
axes[1].set_title("Separabilidad de Especies según Pétalos")
axes[1].set_xlabel("Longitud del pétalo (cm)")
axes[1].set_ylabel("Ancho del pétalo (cm)")
axes[1].legend(title="Especie")

plt.tight_layout()
plt.show()

### 1.c. Funcionamiento Intuitivo de KNN ($k = 3$) y Optimización de Hiperparámetros
El algoritmo **k-Nearest Neighbors (KNN)** es un método supervisado *no paramétrico* basado en instancias. Para clasificar una nueva flor $\mathbf{x}_{\text{nueva}}$:
1. Calcula la **distancia euclidiana** respecto a todas las flores del conjunto de entrenamiento:
   $$d(\mathbf{x}, \mathbf{z}) = \sqrt{\sum_{j=1}^{p} (x_j - z_j)^2}$$
2. Selecciona los **$k$ vecinos más cercanos** (en el modelo base de la pauta, $k = 3$).
3. Asigna la especie por **votación mayoritaria** entre esos $3$ vecinos (o ponderada por el inverso de la distancia en su variante optimizada).

### 1.d. Evaluación de Clasificación y Mejora Autónoma (`Nivel Sobresaliente`)
Para evaluar el rendimiento se emplea inicialmente `accuracy_score` (proporción de predicciones correctas sobre el total), y se profundiza mediante **Precisión (*Precision*)**, **Sensibilidad (*Recall*)**, **F1-Score**, **Matriz de Confusión** y **Validación Cruzada Estratificada de 5 pliegues (`StratifiedKFold`)**:
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}, \qquad \text{Precision} = \frac{TP}{TP + FP}, \qquad \text{Recall} = \frac{TP}{TP + FN}, \qquad F_1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

Además del modelo base `KNN (k=3)`, se implementa una optimización con `Pipeline(StandardScaler + KNeighborsClassifier)` y `GridSearchCV` explorando $k \in [1, 15]$, ponderación (`uniform` vs. `distance`) y métrica (`euclidean` vs. `manhattan`), comparando su efectividad frente a **Random Forest** y **Support Vector Machine (SVM con kernel RBF)**.

In [ ]:
# 2. División estratificada entrenamiento / prueba (80% / 20%)
X_iris = iris.data
y_iris = iris.target

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.20, random_state=RANDOM_STATE, stratify=y_iris
)
print(f"Conjunto Train: {X_train_i.shape[0]} muestras | Conjunto Test: {X_test_i.shape[0]} muestras")

# 3. Modelo Base (requisito pauta): KNN con k = 3
knn_base = KNeighborsClassifier(n_neighbors=3)
knn_base.fit(X_train_i, y_train_i)
y_pred_knn_base = knn_base.predict(X_test_i)

print(f"\n[Modelo Base] Accuracy KNN (k=3) en Test: {accuracy_score(y_test_i, y_pred_knn_base):.4f}")
print(classification_report(y_test_i, y_pred_knn_base, target_names=iris.target_names, digits=4))

# 4. Mejora Autónoma: Pipeline (StandardScaler) + GridSearchCV con StratifiedKFold(5)
cv_iris = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pipe_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])
param_grid_knn = {
    "knn__n_neighbors": list(range(1, 16)),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"],
}
grid_knn = GridSearchCV(pipe_knn, param_grid_knn, cv=cv_iris, scoring="f1_macro", n_jobs=-1)
grid_knn.fit(X_train_i, y_train_i)
y_pred_knn_opt = grid_knn.predict(X_test_i)

print("Mejores hiperparámetros KNN (GridSearchCV):", grid_knn.best_params_)
print(f"Mejor F1-Macro en Validación Cruzada (Train CV-5): {grid_knn.best_score_:.4f}")
print(f"\n[Modelo Optimizado] Accuracy KNN Optimizado en Test: {accuracy_score(y_test_i, y_pred_knn_opt):.4f}")
print(classification_report(y_test_i, y_pred_knn_opt, target_names=iris.target_names, digits=4))

# Curva de rendimiento CV-5 según k (para visualización)
k_range = list(range(1, 16))
k_cv_scores = []
for k in k_range:
    pipe_k = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=k))])
    scores_k = cross_val_score(pipe_k, X_train_i, y_train_i, cv=cv_iris, scoring="f1_macro")
    k_cv_scores.append(scores_k.mean())

# 5. Comparación de modelos alternativos
modelos_cls = {
    "KNN Base (k=3)": KNeighborsClassifier(n_neighbors=3),
    "KNN Optimizado (Pipeline+GS)": grid_knn.best_estimator_,
    "Random Forest (100 árboles)": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "SVM (RBF + StandardScaler)": Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", random_state=RANDOM_STATE)),
    ]),
}

resultados_cls = []
for nombre, modelo in modelos_cls.items():
    modelo.fit(X_train_i, y_train_i)
    pred = modelo.predict(X_test_i)
    cv_acc = cross_val_score(modelo, X_iris, y_iris, cv=cv_iris, scoring="accuracy")
    cv_f1 = cross_val_score(modelo, X_iris, y_iris, cv=cv_iris, scoring="f1_macro")
    resultados_cls.append({
        "Modelo": nombre,
        "Accuracy_Test": accuracy_score(y_test_i, pred),
        "Precision_Macro": precision_score(y_test_i, pred, average="macro"),
        "Recall_Macro": recall_score(y_test_i, pred, average="macro"),
        "F1_Macro_Test": f1_score(y_test_i, pred, average="macro"),
        "CV5_Acc_Media": cv_acc.mean(),
        "CV5_Acc_Std": cv_acc.std(),
        "CV5_F1_Media": cv_f1.mean(),
    })

df_res_cls = pd.DataFrame(resultados_cls)
print("\n=== Comparativa Consolidada de Modelos de Clasificación (Iris Dataset) ===")
print(df_res_cls.round(4).to_string(index=False))

# 6. Reducción de dimensiones con PCA 2D sobre variables estandarizadas
X_iris_std = StandardScaler().fit_transform(X_iris)
pca_iris = PCA(n_components=2, random_state=RANDOM_STATE)
X_iris_pca = pca_iris.fit_transform(X_iris_std)
var_exp = pca_iris.explained_variance_ratio_
print(
    f"\nPCA 2D Iris -> PC1: {var_exp[0]*100:.2f}% | PC2: {var_exp[1]*100:.2f}% "
    f"| Varianza Acumulada: {var_exp.sum()*100:.2f}%"
)

# Figura 2: Curva k vs F1-Macro, Matriz de Confusión y Proyección PCA 2D
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

axes[0].plot(k_range, k_cv_scores, marker="o", linewidth=2, color="#1f4e79")
k_opt_curve = k_range[int(np.argmax(k_cv_scores))]
axes[0].axvline(k_opt_curve, color="#c0392b", linestyle="--", label=f"Mejor k = {k_opt_curve}")
axes[0].set_title("Selección de k en KNN (CV-5 F1-Macro)")
axes[0].set_xlabel("Número de vecinos (k)")
axes[0].set_ylabel("F1-Macro Promedio (Train CV-5)")
axes[0].set_xticks(k_range)
axes[0].legend()

ConfusionMatrixDisplay.from_predictions(
    y_test_i,
    y_pred_knn_opt,
    display_labels=iris.target_names,
    cmap="Blues",
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("Matriz de Confusión — KNN Optimizado (Test)")

for idx, sp in enumerate(iris.target_names):
    axes[2].scatter(
        X_iris_pca[y_iris == idx, 0],
        X_iris_pca[y_iris == idx, 1],
        label=sp,
        alpha=0.85,
        s=55,
    )
axes[2].set_title("Proyección PCA 2D — Iris Dataset")
axes[2].set_xlabel(f"PC1 ({var_exp[0]*100:.1f}% varianza)")
axes[2].set_ylabel(f"PC2 ({var_exp[1]*100:.1f}% varianza)")
axes[2].legend(title="Especie")

plt.tight_layout()
plt.show()

---
## Desafío 2: Clustering No Supervisado — *Mall Customers Dataset*

### 2.a. Exploración de los Datos (`EDA`) y Variables Clave
El conjunto **Mall Customers** registra información demográfica y de comportamiento comercial de **$200$ clientes** de un centro comercial a través de $5$ atributos:

| Variable | Tipo | Descripción | Uso en Clustering |
| :--- | :---: | :--- | :--- |
| `CustomerID` | Identificador | Código único secuencial (`1` a `200`) | **Excluido del modelado:** es un índice arbitrario sin significado geométrico. |
| `Gender` | Categórica | Género del cliente (`Male` / `Female`) | Análisis descriptivo y perfilamiento posterior de los segmentos. |
| `Age` | Numérica | Edad en años | Incluida en la extensión multivariada 3D de segmentación. |
| `Annual Income (k$)` | Numérica | Ingreso anual estimado en miles de dólares | **Variable principal eje X** para la segmentación 2D. |
| `Spending Score (1-100)` | Numérica | Puntaje asignado por el centro comercial según hábito y volumen de gasto (`1` a `100`) | **Variable principal eje Y** para la segmentación 2D. |

**¿Por qué estandarizar antes de aplicar K-Means?** Aunque ambas variables principales presentan órdenes de magnitud cercanos, `Annual Income (k$)` alcanza valores superiores a $130$ mientras `Spending Score` llega a $100$ (y `Age` oscila entre $18$ y $70$). Aplicar `StandardScaler` asegura que cada variable contribuya con igual peso ($\mu = 0, \sigma = 1$) al cálculo de distancias euclidianas.

In [ ]:
# 1. Carga y auditoría inicial del Mall Customers Dataset
df_mall = cargar_mall_customers()

print("Tipos de datos:\n", df_mall.dtypes)
print("\nValores nulos por columna:\n", df_mall.isnull().sum().to_dict())
print("Filas duplicadas:", df_mall.duplicated().sum())

print("\nDistribución por género:")
gender_counts = df_mall["Gender"].value_counts()
for g, cnt in gender_counts.items():
    print(f" - {g:<8}: {cnt} clientes ({cnt / len(df_mall) * 100:.1f}%)")

mall_num_cols = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
print("\nEstadísticas descriptivas:")
print(df_mall[mall_num_cols].describe().round(2))

# Figura 3: Panel EDA 2x2 de Mall Customers
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.histplot(df_mall["Annual Income (k$)"], bins=18, kde=True, color="#2b5c8f", ax=axes[0, 0])
axes[0, 0].set_title("Distribución del Ingreso Anual (k$)")
axes[0, 0].set_xlabel("Annual Income (k$)")

sns.histplot(df_mall["Spending Score (1-100)"], bins=18, kde=True, color="#1b8a6b", ax=axes[0, 1])
axes[0, 1].set_title("Distribución del Puntaje de Gasto (1-100)")
axes[0, 1].set_xlabel("Spending Score (1-100)")

sns.scatterplot(
    data=df_mall,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Gender",
    style="Gender",
    s=70,
    alpha=0.85,
    ax=axes[1, 0],
)
axes[1, 0].set_title("Ingreso Anual vs. Puntaje de Gasto por Género")

sns.heatmap(df_mall[mall_num_cols].corr(), annot=True, cmap="Blues", fmt=".2f", vmin=-1, vmax=1, ax=axes[1, 1])
axes[1, 1].set_title("Correlación entre Variables Numéricas")

plt.tight_layout()
plt.show()

### 2.b y 2.c. Aplicación de K-Means ($k = 2, 3$), Cálculo de Distancias y Evaluación
El algoritmo **K-Means** agrupa las observaciones buscando $k$ prototipos llamados **centroides** ($\mu_1, \dots, \mu_k$) mediante un proceso iterativo:
1. Inicializa $k$ centroides en el espacio de características.
2. **Paso de asignación:** Asigna cada cliente $x_i$ al centroide más cercano según la **distancia euclidiana**:
   $$d(x_i, \mu_j) = \|x_i - \mu_j\|_2$$
3. **Paso de actualización:** Recalcula cada centroide como el promedio de los puntos asignados a su grupo hasta estabilizar la **Inercia o WCSS** (*Within-Cluster Sum of Squares*):
   $$\text{WCSS} = \sum_{j=1}^{k} \sum_{x_i \in C_j} \|x_i - \mu_j\|^2$$

* **¿Por qué evaluar más allá de $k = 2$ o $k = 3$?** Aunque $k = 2$ y $k = 3$ capturan divisiones gruesas (por ejemplo, gasto alto vs. bajo), dejan mezclados perfiles comerciales opuestos en los cuadrantes de ingreso.
* **Optimización y algoritmos alternativos (`Nivel Sobresaliente`):** Se aplica el **Método del Codo** junto con dos métricas objetivas de cohesión y separación:
  * **Índice de Silueta (`Silhouette Score`)** $\in [-1, 1]$ (valores más cercanos a $+1$ indican clusters compactos y bien separados):
    $$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$$
  * **Índice de Davies-Bouldin** (valores más cercanos a $0$ indican menor solapamiento entre clusters).
  
  Con base en estas métricas se determina el $k$ óptimo ($k = 5$), se compara contra **Clustering Jerárquico Aglomerativo (enlace Ward + Dendrograma)** y **DBSCAN** (basado en densidad), se construye un perfilamiento comercial por segmento y se añade una extensión multivariada 3D (`Age`, `Annual Income`, `Spending Score`) visualizada con **PCA 2D**.

In [ ]:
# Preparación y escalamiento de características principales 2D
cluster_cols = ["Annual Income (k$)", "Spending Score (1-100)"]
X_mall_raw = df_mall[cluster_cols].values
scaler_mall = StandardScaler()
X_mall_scaled = scaler_mall.fit_transform(X_mall_raw)

# 1. Modelos Base solicitados por pauta: K-Means con k = 2 y k = 3
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
base_k_metrics = []

for ax, k in zip(axes, [2, 3]):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_mall_scaled)
    centroids_orig = scaler_mall.inverse_transform(km.cluster_centers_)
    sil = silhouette_score(X_mall_scaled, labels)
    db = davies_bouldin_score(X_mall_scaled, labels)
    base_k_metrics.append({"k": k, "WCSS": km.inertia_, "Silhouette": sil, "Davies_Bouldin": db})

    scatter = ax.scatter(
        X_mall_raw[:, 0], X_mall_raw[:, 1], c=labels, cmap="tab10", s=55, alpha=0.85, edgecolors="k", linewidth=0.3
    )
    ax.scatter(
        centroids_orig[:, 0], centroids_orig[:, 1], c="red", marker="X", s=220, label="Centroides", edgecolors="black"
    )
    ax.set_title(f"K-Means Base (k = {k}) | Silueta = {sil:.3f}")
    ax.set_xlabel("Annual Income (k$)")
    ax.set_ylabel("Spending Score (1-100)")
    ax.legend()

plt.tight_layout()
plt.show()

print("=== Métricas K-Means Base (k=2 y k=3) ===")
print(pd.DataFrame(base_k_metrics).round(4).to_string(index=False))

# 2. Búsqueda sistemática del k óptimo (k = 2..10): Codo, Silueta y Davies-Bouldin
k_candidates = list(range(2, 11))
eval_k = []
for k in k_candidates:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    lbl_k = km_k.fit_predict(X_mall_scaled)
    eval_k.append({
        "k": k,
        "WCSS": km_k.inertia_,
        "Silhouette": silhouette_score(X_mall_scaled, lbl_k),
        "Davies_Bouldin": davies_bouldin_score(X_mall_scaled, lbl_k),
    })

df_k_eval = pd.DataFrame(eval_k)
k_opt = int(df_k_eval.loc[df_k_eval["Silhouette"].idxmax(), "k"])
print("\n=== Evaluación de Número de Clusters (k = 2 a 10) ===")
print(df_k_eval.round(4).to_string(index=False))
print(f"Número óptimo de clusters según máximo Índice de Silueta: k = {k_opt}")

# Figura 5: Método del Codo + Índice de Silueta
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

axes[0].plot(df_k_eval["k"], df_k_eval["WCSS"], marker="o", linewidth=2, color="#1f4e79")
axes[0].axvline(k_opt, color="#c0392b", linestyle="--", label=f"Codo óptimo (k = {k_opt})")
axes[0].set_title("Método del Codo — Inercia (WCSS)")
axes[0].set_xlabel("Número de clusters (k)")
axes[0].set_ylabel("WCSS (datos escalados)")
axes[0].set_xticks(k_candidates)
axes[0].legend()

axes[1].plot(df_k_eval["k"], df_k_eval["Silhouette"], marker="s", linewidth=2, color="#1b8a6b")
axes[1].axvline(k_opt, color="#c0392b", linestyle="--", label=f"Máxima Silueta (k = {k_opt})")
axes[1].set_title("Índice de Silueta por Número de Clusters")
axes[1].set_xlabel("Número de clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_xticks(k_candidates)
axes[1].legend()

plt.tight_layout()
plt.show()

# 3. Comparación entre K-Means Óptimo, Clustering Jerárquico Ward y DBSCAN
km_opt = KMeans(n_clusters=k_opt, n_init=10, random_state=RANDOM_STATE)
lbl_km_opt = km_opt.fit_predict(X_mall_scaled)

hc_ward = AgglomerativeClustering(n_clusters=k_opt, linkage="ward")
lbl_ward = hc_ward.fit_predict(X_mall_scaled)

dbscan = DBSCAN(eps=0.35, min_samples=5)
lbl_db = dbscan.fit_predict(X_mall_scaled)


def resumen_clustering(nombre: str, X_sc: np.ndarray, labels: np.ndarray, wcss=np.nan) -> dict:
    mask_valid = labels != -1
    n_noise = int((labels == -1).sum())
    n_clusters = len(set(labels[mask_valid]))
    if n_clusters >= 2 and mask_valid.sum() > n_clusters:
        sil = silhouette_score(X_sc[mask_valid], labels[mask_valid])
        db = davies_bouldin_score(X_sc[mask_valid], labels[mask_valid])
    else:
        sil, db = np.nan, np.nan
    return {
        "Algoritmo": nombre,
        "Clusters_Detectados": n_clusters,
        "Puntos_Ruido": n_noise,
        "WCSS": wcss,
        "Silhouette": sil,
        "Davies_Bouldin": db,
    }


df_res_clu = pd.DataFrame([
    resumen_clustering("K-Means Base (k=2)", X_mall_scaled, KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE).fit_predict(X_mall_scaled), base_k_metrics[0]["WCSS"]),
    resumen_clustering("K-Means Base (k=3)", X_mall_scaled, KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit_predict(X_mall_scaled), base_k_metrics[1]["WCSS"]),
    resumen_clustering(f"K-Means Óptimo (k={k_opt})", X_mall_scaled, lbl_km_opt, km_opt.inertia_),
    resumen_clustering(f"Jerárquico Ward (k={k_opt})", X_mall_scaled, lbl_ward),
    resumen_clustering("DBSCAN (eps=0.35, min=5)", X_mall_scaled, lbl_db),
])
print("\n=== Comparativa de Algoritmos de Clustering (Mall Customers 2D) ===")
print(df_res_clu.round(4).to_string(index=False))

# Asignación dinámica de nombres de negocio según coordenadas reales de los centroides
centroids_opt_orig = scaler_mall.inverse_transform(km_opt.cluster_centers_)


def nombrar_segmentos_dinamicos(centroids: np.ndarray, X_orig: np.ndarray) -> dict:
    mean_inc, mean_spd = X_orig[:, 0].mean(), X_orig[:, 1].mean()
    std_inc, std_spd = X_orig[:, 0].std(), X_orig[:, 1].std()
    mapping = {}
    usados = set()

    if len(centroids) == 5:
        dist_centro = np.sqrt(((centroids[:, 0] - mean_inc) / std_inc) ** 2 + ((centroids[:, 1] - mean_spd) / std_spd) ** 2)
        idx_centro = int(np.argmin(dist_centro))
        mapping[idx_centro] = "Estándar / Moderado"
        usados.add("Estándar / Moderado")

    for cid, (inc, spd) in enumerate(centroids):
        if cid in mapping:
            continue
        if inc >= mean_inc and spd >= mean_spd:
            etiqueta = "VIP / Objetivo (Alto Ingreso, Alto Gasto)"
        elif inc >= mean_inc and spd < mean_spd:
            etiqueta = "Conservador (Alto Ingreso, Bajo Gasto)"
        elif inc < mean_inc and spd >= mean_spd:
            etiqueta = "Impulsivo / Entusiasta (Bajo Ingreso, Alto Gasto)"
        else:
            etiqueta = "Austero (Bajo Ingreso, Bajo Gasto)"

        if etiqueta in usados:
            etiqueta = f"{etiqueta} [{cid}]"
        mapping[cid] = etiqueta
        usados.add(etiqueta)
    return mapping


segment_map = nombrar_segmentos_dinamicos(centroids_opt_orig, X_mall_raw)
df_mall["Cluster_Opt"] = lbl_km_opt
df_mall["Segmento"] = df_mall["Cluster_Opt"].map(segment_map)

# Figura 6: K-Means Óptimo, Dendrograma Ward y Jerárquico Ward
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.scatterplot(
    data=df_mall,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Segmento",
    palette="tab10",
    s=65,
    alpha=0.88,
    ax=axes[0],
)
axes[0].scatter(
    centroids_opt_orig[:, 0], centroids_opt_orig[:, 1], c="black", marker="X", s=200, label="Centroides"
)
axes[0].set_title(f"K-Means Óptimo (k = {k_opt}) — Segmentos Comerciales")
axes[0].legend(fontsize=7.5, loc="best")

Z_ward = linkage(X_mall_scaled, method="ward")
dendrogram(Z_ward, truncate_mode="lastp", p=20, leaf_rotation=90.0, leaf_font_size=8.0, ax=axes[1])
axes[1].set_title("Dendrograma — Enlace Ward (Últimas 20 fusiones)")
axes[1].set_xlabel("Tamaño de subcluster")
axes[1].set_ylabel("Distancia Euclidiana")

axes[2].scatter(
    X_mall_raw[:, 0], X_mall_raw[:, 1], c=lbl_ward, cmap="tab10", s=60, alpha=0.88, edgecolors="k", linewidth=0.3
)
axes[2].set_title(f"Clustering Jerárquico Ward (k = {k_opt})")
axes[2].set_xlabel("Annual Income (k$)")
axes[2].set_ylabel("Spending Score (1-100)")

plt.tight_layout()
plt.show()

# Perfilamiento comercial por segmento
perfil_mall = (
    df_mall.assign(Es_Femenino=(df_mall["Gender"] == "Female").astype(int))
    .groupby("Segmento")
    .agg(
        Clientes=("CustomerID", "count"),
        Edad_Media=("Age", "mean"),
        Ingreso_Medio_k=("Annual Income (k$)", "mean"),
        Gasto_Medio=("Spending Score (1-100)", "mean"),
        Pct_Femenino=("Es_Femenino", lambda s: s.mean() * 100),
    )
    .round(2)
)
perfil_mall["Porcentaje_%"] = (perfil_mall["Clientes"] / len(df_mall) * 100).round(1)
print("\n=== Perfilamiento Comercial de los Segmentos Obtenidos ===")
print(perfil_mall.sort_values("Ingreso_Medio_k", ascending=False).to_string())

# 4. Extensión Multivariada 3D (Age + Annual Income + Spending Score) + PCA 2D
X_mall_3d = StandardScaler().fit_transform(df_mall[mall_num_cols])
sil_3d = {k: silhouette_score(X_mall_3d, KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_mall_3d)) for k in range(2, 9)}
k_3d_opt = max(sil_3d, key=sil_3d.get)
lbl_3d = KMeans(n_clusters=k_3d_opt, n_init=10, random_state=RANDOM_STATE).fit_predict(X_mall_3d)
pca_mall = PCA(n_components=2, random_state=RANDOM_STATE)
X_mall_pca = pca_mall.fit_transform(X_mall_3d)

print(
    f"\n[Extensión 3D con Edad] Mejor k = {k_3d_opt} (Silhouette 3D = {sil_3d[k_3d_opt]:.4f}) | "
    f"Varianza retenida PCA 2D = {pca_mall.explained_variance_ratio_.sum()*100:.2f}%"
)

---
## Desafío 3: Regresión Predictiva — *Boston Housing Dataset*

### 3.a. Exploración de los Datos (`EDA`) y Variables Clave
El conjunto **Boston Housing** (Harrison & Rubinfeld, 1978) reúne **$506$ zonas censales** del área metropolitana de Boston descritas mediante **$13$ variables predictoras** socioeconómicas, estructurales y ambientales, junto con la variable objetivo **`MEDV`** (*valor mediano de viviendas ocupadas por sus propietarios, en miles de USD*):

| Variable | Descripción | Relevancia Analítica |
| :--- | :--- | :--- |
| **`RM`** | Promedio de habitaciones por vivienda | **Variable clave de la regresión simple:** mantiene una fuerte correlación positiva con `MEDV`. |
| **`LSTAT`** | Porcentaje de población de estatus socioeconómico bajo | **Predictor inverso dominante:** presenta una relación negativa intensa y no lineal con `MEDV`. |
| **`PTRATIO`** | Ratio alumno-profesor por distrito escolar | Correlación negativa moderada con el valor inmobiliario. |
| **`CRIM`, `INDUS`, `NOX`, `AGE`, `DIS`, `RAD`, `TAX`, `ZN`, `CHAS`, `B`** | Criminalidad, uso industrial, óxidos nítricos ($\text{NO}_x$), antigüedad, distancia a centros de empleo, accesibilidad a autopistas, impuesto predial, zonificación, proximidad al río Charles y proporción demográfica histórica | Variables multivariadas complementarias que explican heterogeneidad espacial y socioeconómica. |
| **`MEDV`** *(Target)* | Valor mediano de la vivienda en miles de USD (`$1.000`) | Variable objetivo continua; presenta censura superior (*top-coding*) en `50.0` (`$50.000`). |

In [ ]:
# 1. Carga y auditoría de calidad del Boston Housing Dataset
df_boston = cargar_boston_housing()

print("Valores nulos totales:", int(df_boston.isnull().sum().sum()))
print("Filas duplicadas:", int(df_boston.duplicated().sum()))
censurados_50 = int((df_boston["MEDV"] == 50.0).sum())
print(f"Registros con censura superior (MEDV == 50.0): {censurados_50} ({censurados_50 / len(df_boston) * 100:.2f}%)")

print("\nEstadísticas descriptivas de variables clave (RM, LSTAT, PTRATIO, CRIM, NOX, MEDV):")
print(df_boston[["RM", "LSTAT", "PTRATIO", "CRIM", "NOX", "MEDV"]].describe().round(3))

corr_medv = df_boston.corr()["MEDV"].drop("MEDV").sort_values(key=lambda s: s.abs(), ascending=False)
print("\nCorrelación de Pearson de cada predictor con MEDV (ordenado por magnitud):")
print(corr_medv.round(4))

# Figura 7: Heatmap top correlaciones + RM vs MEDV + LSTAT vs MEDV
top_vars = list(corr_medv.head(7).index) + ["MEDV"]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.heatmap(df_boston[top_vars].corr(), annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Correlación — Top Variables con MEDV")

sns.regplot(
    data=df_boston, x="RM", y="MEDV", scatter_kws={"alpha": 0.55, "s": 28}, line_kws={"color": "#c0392b", "linewidth": 2}, ax=axes[1]
)
axes[1].set_title("Habitaciones (RM) vs. Precio (MEDV)")
axes[1].set_xlabel("Promedio de habitaciones (RM)")
axes[1].set_ylabel("MEDV (miles de USD)")

sns.regplot(
    data=df_boston, x="LSTAT", y="MEDV", order=2, scatter_kws={"alpha": 0.55, "s": 28, "color": "#1b8a6b"}, line_kws={"color": "#1f4e79", "linewidth": 2}, ax=axes[2]
)
axes[2].set_title("Estatus Bajo (LSTAT) vs. Precio (MEDV)")
axes[2].set_xlabel("% población LSTAT")
axes[2].set_ylabel("MEDV (miles de USD)")

plt.tight_layout()
plt.show()

### 3.b. Regresión Lineal Simple (`RM -> MEDV`) y Modelado Avanzado
La **Regresión Lineal Simple** ajusta una recta óptima que minimiza la suma de los errores cuadráticos entre el valor real $y_i$ y el valor predicho $\hat{y}_i$:
$$\widehat{\text{MEDV}} = \beta_0 + \beta_1 \cdot \text{RM}$$
donde $\beta_0$ es el intercepto y $\beta_1$ representa cuánto aumenta (en miles de dólares) el valor esperado de la vivienda por cada habitación adicional.

### 3.c. Métricas de Evaluación en Regresión ($R^2$, $R^2_{\text{adj}}$, MAE, MSE y RMSE)
* **Coeficiente de Determinación ($R^2$):** Mide la proporción de variabilidad del precio explicada por el modelo. Un valor cercano a **$1.0$** indica excelente ajuste; $R^2 = 0$ equivale a predecir simplemente la media:
  $$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2}, \qquad R^2_{\text{adj}} = 1 - (1 - R^2)\frac{n - 1}{n - p - 1}$$
* **MAE, MSE y RMSE:** Miden la magnitud del error en la escala de los datos (RMSE conserva las mismas unidades de `MEDV`, es decir, miles de dólares; un RMSE menor indica mayor precisión predictiva):
  $$\text{MAE} = \frac{1}{n}\sum_{i=1}^n |y_i - \hat{y}_i|, \qquad \text{MSE} = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2, \qquad \text{RMSE} = \sqrt{\text{MSE}}$$

Para alcanzar el **Nivel Sobresaliente**, se compara el modelo base (`LinearRegression` solo con `RM`) frente a **Regresión Lineal Múltiple** ($13$ predictores), **Ridge (`L2`)**, **Lasso (`L1`)** y un ensamble no lineal **Random Forest Regressor** optimizado mediante `GridSearchCV`, todos encapsulados en `Pipeline` con `SimpleImputer(strategy='median')`.

In [ ]:
# 2. División entrenamiento / prueba (80% / 20%)
X_boston = df_boston.drop(columns=["MEDV"])
y_boston = df_boston["MEDV"]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_boston, y_boston, test_size=0.20, random_state=RANDOM_STATE
)
cv_boston = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def evaluar_regresor(nombre: str, modelo, X_tr, y_tr, X_te, y_te, cv, n_features: int) -> tuple:
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_te)
    r2 = r2_score(y_te, y_pred)
    r2_adj = 1.0 - (1.0 - r2) * (len(y_te) - 1) / (len(y_te) - n_features - 1)
    mae = mean_absolute_error(y_te, y_pred)
    mse = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    cv_r2 = cross_val_score(modelo, X_tr, y_tr, cv=cv, scoring="r2", n_jobs=-1)
    resumen = {
        "Modelo": nombre,
        "R2_Test": r2,
        "R2_Ajustado": r2_adj,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "Error_Medio_USD": rmse * 1000.0,
        "CV5_R2_Media": cv_r2.mean(),
        "CV5_R2_Std": cv_r2.std(),
    }
    return resumen, y_pred, modelo


# Modelo 1 (Base pauta): Regresión Lineal Simple con RM
pipe_lr_simple = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("lr", LinearRegression()),
])
res_lr_s, pred_lr_s, mod_lr_s = evaluar_regresor(
    "Regresión Lineal Simple (RM)", pipe_lr_simple, X_train_b[["RM"]], y_train_b, X_test_b[["RM"]], y_test_b, cv_boston, 1
)
b0 = mod_lr_s.named_steps["lr"].intercept_
b1 = mod_lr_s.named_steps["lr"].coef_[0]
print(f"Ecuación Regresión Lineal Simple: MEDV_hat = {b0:.4f} + ({b1:.4f}) * RM")

# Modelo 2: Regresión Lineal Múltiple (13 variables)
pipe_lr_mult = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lr", LinearRegression()),
])
res_lr_m, pred_lr_m, mod_lr_m = evaluar_regresor(
    "Regresión Lineal Múltiple (13 vars)", pipe_lr_mult, X_train_b, y_train_b, X_test_b, y_test_b, cv_boston, X_train_b.shape[1]
)

# Modelo 3: RidgeCV (Regularización L2)
pipe_ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 50))),
])
res_ridge, pred_ridge, mod_ridge = evaluar_regresor(
    "Ridge Regression (CV)", pipe_ridge, X_train_b, y_train_b, X_test_b, y_test_b, cv_boston, X_train_b.shape[1]
)
print(f"Alpha óptimo RidgeCV: {mod_ridge.named_steps['ridge'].alpha_:.4f}")

# Modelo 4: LassoCV (Regularización L1 y selección de variables)
pipe_lasso = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=cv_boston, random_state=RANDOM_STATE, max_iter=10000)),
])
res_lasso, pred_lasso, mod_lasso = evaluar_regresor(
    "Lasso Regression (CV)", pipe_lasso, X_train_b, y_train_b, X_test_b, y_test_b, cv_boston, X_train_b.shape[1]
)
lasso_coefs = mod_lasso.named_steps["lasso"].coef_
print(f"Alpha óptimo LassoCV: {mod_lasso.named_steps['lasso'].alpha_:.4f} | Variables activas: {int((lasso_coefs != 0).sum())}/{len(lasso_coefs)}")

# Modelo 5: Random Forest Regressor optimizado con GridSearchCV
pipe_rf_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestRegressor(random_state=RANDOM_STATE)),
])
param_grid_rf = {
    "rf__n_estimators": [100, 200],
    "rf__max_depth": [None, 10, 15],
    "rf__min_samples_split": [2, 5],
}
grid_rf = GridSearchCV(pipe_rf_reg, param_grid_rf, cv=cv_boston, scoring="r2", n_jobs=-1)
grid_rf.fit(X_train_b, y_train_b)
print("Mejores hiperparámetros Random Forest Regressor:", grid_rf.best_params_)

res_rf, pred_rf, mod_rf = evaluar_regresor(
    "Random Forest Regressor (Optimizado)", grid_rf.best_estimator_, X_train_b, y_train_b, X_test_b, y_test_b, cv_boston, X_train_b.shape[1]
)

df_res_reg = pd.DataFrame([res_lr_s, res_lr_m, res_ridge, res_lasso, res_rf])
print("\n=== Comparativa Consolidada de Modelos de Regresión (Boston Housing) ===")
print(df_res_reg.round(4).to_string(index=False))

# Figura 8: Ajuste Lineal Simple y Real vs. Predicho Comparativo
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

rm_grid = np.linspace(X_test_b["RM"].min(), X_test_b["RM"].max(), 100)
axes[0].scatter(X_test_b["RM"], y_test_b, alpha=0.75, color="#1f4e79", label="Datos reales (Test)")
axes[0].plot(rm_grid, b0 + b1 * rm_grid, color="#c0392b", linewidth=2.5, label=f"Recta OLS (R² = {res_lr_s['R2_Test']:.3f})")
axes[0].set_title("Regresión Lineal Simple: RM vs. MEDV")
axes[0].set_xlabel("Promedio de habitaciones (RM)")
axes[0].set_ylabel("MEDV (miles de USD)")
axes[0].legend()

axes[1].scatter(y_test_b, pred_lr_s, alpha=0.55, label=f"Lineal Simple (RMSE={res_lr_s['RMSE']:.2f})", marker="^")
axes[1].scatter(y_test_b, pred_lr_m, alpha=0.60, label=f"Lineal Múltiple (RMSE={res_lr_m['RMSE']:.2f})", marker="s")
axes[1].scatter(y_test_b, pred_rf, alpha=0.75, label=f"Random Forest (RMSE={res_rf['RMSE']:.2f})", marker="o")
lims = [min(y_test_b.min(), pred_lr_m.min()) - 2, max(y_test_b.max(), pred_rf.max()) + 2]
axes[1].plot(lims, lims, "k--", linewidth=1.5, label="Predicción ideal (y = ŷ)")
axes[1].set_title("Valores Reales vs. Predichos en Test")
axes[1].set_xlabel("MEDV Real (miles de USD)")
axes[1].set_ylabel("MEDV Predicho (miles de USD)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Figura 9: Análisis de Residuos y Feature Importance
residuos_lr = y_test_b - pred_lr_m
residuos_rf = y_test_b - pred_rf
importances = pd.Series(mod_rf.named_steps["rf"].feature_importances_, index=X_boston.columns).sort_values(ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].scatter(pred_lr_m, residuos_lr, alpha=0.6, label="Lineal Múltiple", color="#e67e22")
axes[0].scatter(pred_rf, residuos_rf, alpha=0.65, label="Random Forest", color="#1b8a6b")
axes[0].axhline(0, color="black", linestyle="--", linewidth=1.2)
axes[0].set_title("Residuos vs. Valores Predichos")
axes[0].set_xlabel("MEDV Predicho")
axes[0].set_ylabel("Residuo (Real - Predicho)")
axes[0].legend()

sns.histplot(residuos_rf, bins=18, kde=True, color="#1b8a6b", ax=axes[1])
axes[1].axvline(0, color="#c0392b", linestyle="--")
axes[1].set_title("Distribución de Residuos — Random Forest")
axes[1].set_xlabel("Error de predicción (miles de USD)")

importances.tail(10).plot(kind="barh", color="#1f4e79", ax=axes[2])
axes[2].set_title("Top 10 Importancia de Variables (Random Forest)")
axes[2].set_xlabel("Importancia relativa (reducción MSE)")

plt.tight_layout()
plt.show()

---
## Desafío 4: Evaluación Consolidada del Rendimiento y Efectividad Global

Tal como distingue el marco teórico de la asignatura (*Apunte N° 2*, IPSS, 2024), **precisión (`accuracy_score`)** y **efectividad de un modelo** no son sinónimos: mientras `Accuracy` mide únicamente la proporción global de aciertos en clasificación, la **efectividad** evalúa integralmente si el modelo cumple su objetivo según el tipo de problema, el balance entre clases y la sensibilidad al tipo de error crítico:

* **4.a. Clasificación (*Iris Dataset*):** `accuracy_score` es válido porque las tres clases están perfectamente balanceadas ($50/50/50$). Aun así, se complementa con `Precision`, `Recall`, `F1-Macro`, Matriz de Confusión y `CV-5` para confirmar que no existan confusiones sistemáticas entre *Versicolor* y *Virginica*.
* **4.b. Clustering (*Mall Customers*):** Al carecer de etiquetas reales, la evaluación combina **inspección visual 2D** (identificando centroides y separación espacial de los $5$ grupos) con métricas geométricas internas (**Inercia WCSS**, **Índice de Silueta** y **Davies-Bouldin**), demostrando por qué $k = 5$ supera objetivamente a $k = 2$ y $k = 3$.
* **4.c. Regresión (*Boston Housing*):** $R^2$ indica la fracción de varianza explicada del precio inmobiliario (valores cercanos a $1$ reflejan mejor ajuste), mientras **RMSE** y **MAE** cuantifican el error real en miles de dólares. El salto desde la regresión simple con `RM` hacia la regresión múltiple y *Random Forest* evidencia el peso de las interacciones no lineales (`LSTAT`, `RM`, `DIS`, `CRIM`).

In [ ]:
# Síntesis cuantitativa comparando Modelo Base vs. Mejor Modelo Optimizado por paradigma
best_cls = df_res_cls.sort_values(["F1_Macro_Test", "CV5_F1_Media"], ascending=False).iloc[0]
base_cls = df_res_cls.loc[df_res_cls["Modelo"] == "KNN Base (k=3)"].iloc[0]

best_clu_row = df_res_clu.dropna(subset=["Silhouette"]).sort_values("Silhouette", ascending=False).iloc[0]
base_clu_row = df_res_clu.loc[df_res_clu["Algoritmo"] == "K-Means Base (k=3)"].iloc[0]

best_reg = df_res_reg.sort_values("RMSE", ascending=True).iloc[0]
base_reg = df_res_reg.loc[df_res_reg["Modelo"] == "Regresión Lineal Simple (RM)"].iloc[0]

tabla_global = pd.DataFrame([
    {
        "Desafío": "1. Clasificación (Iris)",
        "Modelo_Base": base_cls["Modelo"],
        "Metrica_Base": f"Acc={base_cls['Accuracy_Test']:.4f} | CV5-F1={base_cls['CV5_F1_Media']:.4f}",
        "Mejor_Modelo": best_cls["Modelo"],
        "Metrica_Optimizada": f"Acc={best_cls['Accuracy_Test']:.4f} | CV5-F1={best_cls['CV5_F1_Media']:.4f}",
    },
    {
        "Desafío": "2. Clustering (Mall Customers)",
        "Modelo_Base": base_clu_row["Algoritmo"],
        "Metrica_Base": f"Sil={base_clu_row['Silhouette']:.4f} | DB={base_clu_row['Davies_Bouldin']:.4f}",
        "Mejor_Modelo": best_clu_row["Algoritmo"],
        "Metrica_Optimizada": f"Sil={best_clu_row['Silhouette']:.4f} | DB={best_clu_row['Davies_Bouldin']:.4f}",
    },
    {
        "Desafío": "3. Regresión (Boston Housing)",
        "Modelo_Base": base_reg["Modelo"],
        "Metrica_Base": f"R2={base_reg['R2_Test']:.4f} | RMSE={base_reg['RMSE']:.4f}",
        "Mejor_Modelo": best_reg["Modelo"],
        "Metrica_Optimizada": f"R2={best_reg['R2_Test']:.4f} | RMSE={best_reg['RMSE']:.4f}",
    },
])
print("=== Resumen Global de Efectividad: Modelo Base vs. Mejor Modelo ===")
print(tabla_global.to_string(index=False))

# Figura 10: Panel Comparativo Consolidado de los 3 Desafíos
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel A: Clasificación
x_cls = np.arange(len(df_res_cls))
w = 0.35
axes[0].bar(x_cls - w / 2, df_res_cls["Accuracy_Test"], width=w, label="Accuracy Test", color="#1f4e79")
axes[0].bar(x_cls + w / 2, df_res_cls["CV5_F1_Media"], width=w, label="CV-5 F1-Macro", color="#1b8a6b")
axes[0].set_xticks(x_cls)
axes[0].set_xticklabels(["KNN Base", "KNN Opt.", "Rand. Forest", "SVM RBF"], rotation=20, fontsize=8.5)
axes[0].set_ylim(0.8, 1.03)
axes[0].set_title("Clasificación — Iris Dataset")
axes[0].set_ylabel("Puntaje")
axes[0].legend()

# Panel B: Clustering
df_clu_plot = df_res_clu.dropna(subset=["Silhouette"])
axes[1].bar(df_clu_plot["Algoritmo"], df_clu_plot["Silhouette"], color="#2b5c8f")
axes[1].set_title("Clustering — Silhouette Score (Mall)")
axes[1].set_ylabel("Silhouette Score (mayor es mejor)")
axes[1].tick_params(axis="x", rotation=25, labelsize=8)

# Panel C: Regresión
x_reg = np.arange(len(df_res_reg))
ax_r2 = axes[2]
ax_rmse = ax_r2.twinx()
ax_r2.bar(x_reg - 0.18, df_res_reg["R2_Test"], width=0.35, color="#1b8a6b", label="R² Test")
ax_rmse.bar(x_reg + 0.18, df_res_reg["RMSE"], width=0.35, color="#c0392b", alpha=0.85, label="RMSE")
ax_r2.set_xticks(x_reg)
ax_r2.set_xticklabels(["Lin. Simple", "Lin. Múlt.", "Ridge", "Lasso", "Rand. Forest"], rotation=20, fontsize=8.5)
ax_r2.set_title("Regresión — Boston Housing (R² vs RMSE)")
ax_r2.set_ylabel("R² Test", color="#1b8a6b")
ax_rmse.set_ylabel("RMSE (miles USD)", color="#c0392b")
h1, l1 = ax_r2.get_legend_handles_labels()
h2, l2 = ax_rmse.get_legend_handles_labels()
ax_r2.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

---
## Desafío 5: Conclusiones Críticas, Limitaciones y Propuestas de Mejora

### 5.1. Síntesis de Resultados por Paradigma
1. **Clasificación (*Iris Dataset*):** El análisis exploratorio confirmó una fuerte colinealidad entre `petal length` y `petal width`, separando limpiamente a *Iris setosa* mientras *Versicolor* y *Virginica* comparten una frontera estrecha. El modelo base `KNN (k=3)` logra un rendimiento elevado, y la incorporación de `StandardScaler` dentro de un `Pipeline` junto con `GridSearchCV` (`StratifiedKFold`, $k$ óptimo, métrica y ponderación por distancia) y la comparación contra *Random Forest* y *SVM (RBF)* estabilizan la generalización y previenen el *data leakage*. Por su parte, `PCA` en $2$ dimensiones retiene prácticamente toda la varianza explicada del problema.
2. **Clustering (*Mall Customers*):** Aunque la pauta solicita agrupar con `K-Means` en $k = 2$ o $k = 3$, el Método del Codo, el Índice de Silueta y Davies-Bouldin demuestran objetivamente que la estructura geométrica natural en `Annual Income (k$)` y `Spending Score (1-100)` corresponde a **$k = 5$ segmentos comerciales accionables** (*VIP*, *Conservador*, *Estándar*, *Impulsivo* y *Austero*). La validación cruzada frente a **Clustering Jerárquico Ward** confirma la estabilidad de los grupos, mientras **DBSCAN** evidencia sensibilidad ante las variaciones de densidad periférica.
3. **Regresión (*Boston Housing*):** La regresión lineal simple basada exclusivamente en `RM` captura una fracción relevante de la varianza de `MEDV`, pero resulta insuficiente frente a la naturaleza multifactorial y no lineal del mercado inmobiliario. Al incorporar los $13$ predictores (`Regresión Múltiple`, `RidgeCV`, `LassoCV`) y, especialmente, al aplicar `RandomForestRegressor` optimizado con `GridSearchCV`, se reduce drásticamente el `RMSE` y aumenta el $R^2$ al capturar la curvatura no lineal de `LSTAT`, la distancia laboral (`DIS`) y la contaminación (`NOX`).

### 5.2. Limitaciones Identificadas y Propuestas de Mejora
* **Iris Dataset:** Tamaño muestral acotado ($n = 150$, apenas $30$ flores en el conjunto de prueba) que hace sensibles las diferencias de $1$ sola observación. *Mejora:* emplear validación cruzada repetida (`RepeatedStratifiedKFold`) y modelos probabilísticos calibrados para estimar incertidumbre en la frontera *Versicolor–Virginica*.
* **Mall Customers:** Ausencia de variables temporales/transaccionales (**RFM**: Recencia, Frecuencia, Monto) y supuesto de clusters esféricos propio de `K-Means`. *Mejora:* enriquecer con historial de compras y probar **Gaussian Mixture Models (GMM)** o **HDBSCAN** para segmentos con covarianzas elípticas y densidades variables.
* **Boston Housing:** (a) **Censura superior (`MEDV = 50.0`)** en $16$ observaciones que distorsiona los residuos de modelos lineales en el segmento de lujo; (b) **No linealidad** marcada en `LSTAT` y multicolinealidad (`RAD`–`TAX`); (c) **Obsolescencia y sesgo ético** de la variable `B` (razón por la cual `scikit-learn >= 1.2` eliminó `load_boston()`). *Mejora:* aplicar modelos *Gradient Boosting* (`XGBoost` / `HistGradientBoostingRegressor`), transformación logarítmica o regresión censurada (*Tobit*), y migrar en entornos productivos hacia **California Housing** o **Ames Housing**.

---
